<a href="https://colab.research.google.com/github/Rodrigoldarocha/TOTVS_Fundamentos_de_Engenharia_de_Dados_e_Machine_Learning_Portfolio/blob/main/TOTVS_Fundamentos_de_Engenharia_de_Dados_e_Machine_Learning_Portfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TOTVS - Fundamentos de Engenharia de Dados e Machine Learning — Pipeline ETL com Python e IA Generativa

> **Autor:** Rodrigo Rocha
> **Bootcamp:** TOTVS - Fundamentos de Engenharia de Dados e Machine Learning — DIO

---

## O que esse projeto faz?

A ideia aqui é construir um pipeline ETL do zero. ETL significa **Extração, Transformação e Carregamento** — e é um dos fluxos mais usados em projetos de dados no mercado.

Nesse caso específico, vou extrair dados de clientes de um arquivo CSV, usar uma IA generativa para criar mensagens de marketing personalizadas para cada um, e depois salvar tudo em um novo arquivo com os resultados.

O fluxo fica assim:

```
TOTVS 2026 - SDW2023.csv  -->  [EXTRACT]  -->  [TRANSFORM com IA]  -->  [LOAD]  -->  TOTVS 2026 - SDW2023_resultado.csv
```

---

## Por que não usei a API original?

A API pública da Santander Dev Week (`sdw-2023-prd.up.railway.app`) foi descontinuada depois que o evento terminou. Em vez de travar nisso, adaptei o projeto:

- Na etapa de **Extract**: usei um arquivo CSV com os dados dos clientes diretamente
- Na etapa de **Transform**: usei uma função local que simula a IA, gerando mensagens personalizadas por perfil financeiro — sem depender de nenhuma API externa
- Na etapa de **Load**: salvei os resultados localmente em CSV e JSON

O conceito de ETL é exatamente o mesmo. Só mudei as ferramentas.

## Passo 0 — Instalando as dependências

Nessa versão não preciso instalar nenhuma biblioteca externa além do pandas, que já vem no Colab. A etapa de Transformação usa uma função local que gera as mensagens sem chamar nenhuma API.


In [1]:
import pandas as pd
import json

print("Dependências prontas.")

Dependências prontas.


## Passo 1 — Sem necessidade de API Key

Nessa versão, a etapa de Transformação roda localmente. Não preciso de nenhuma chave de API — o que significa que o pipeline funciona sem custo e sem dependência externa.

Isso é útil para estudar e demonstrar o fluxo ETL sem precisar criar conta em nenhum serviço de IA.


In [2]:
# Nenhuma configuração necessária nessa etapa.
# A função de IA vai rodar localmente na etapa de Transformação.

print("Pronto para iniciar o pipeline.")

Pronto para iniciar o pipeline.


---

## E — EXTRACT (Extração)

Essa é a primeira etapa do ETL. O objetivo aqui é obter os dados dos clientes.

No projeto original, essa etapa funcionava assim: tinha um arquivo CSV só com os IDs dos usuários, e o código fazia uma requisição GET para uma API buscando os dados completos de cada um.

Como a API não está mais disponível, adaptei para usar um CSV já com todos os dados necessários. A lógica de extração continua a mesma — estou lendo uma fonte externa de dados e estruturando as informações para a próxima etapa.

### O que cada coluna representa:
- **UserID**: identificador único do cliente
- **Nome**: nome do cliente
- **Saldo**: saldo atual na conta
- **Limite**: limite disponível no cartão
- **Agencia**: número da agência

In [3]:
import pandas as pd

# Criando o arquivo CSV com os dados dos clientes.
# Usei nomes de personagens de anime só para deixar mais divertido,
# mas em um cenário real esses dados viriam de um banco de dados ou sistema interno.
dados_clientes = {
    'UserID':  [1,        2,        3,         4,        5],
    'Nome':    ['Naruto', 'Hinata', 'Sasuke',  'Sakura', 'Kakashi'],
    'Saldo':   [1500.00,  320.50,   8750.00,   220.00,   15000.00],
    'Limite':  [2000.00,  500.00,   10000.00,  1000.00,  20000.00],
    'Agencia': ['0001',   '0001',   '0002',    '0002',   '0003']
}

df_original = pd.DataFrame(dados_clientes)
df_original.to_csv('TOTVS 2026 - SDW2023.csv', index=False)

print("Arquivo TOTVS 2026 - SDW2023.csv criado.")
print()
print(df_original.to_string(index=False))

Arquivo TOTVS 2026 - SDW2023.csv criado.

 UserID    Nome   Saldo  Limite Agencia
      1  Naruto  1500.0  2000.0    0001
      2  Hinata   320.5   500.0    0001
      3  Sasuke  8750.0 10000.0    0002
      4  Sakura   220.0  1000.0    0002
      5 Kakashi 15000.0 20000.0    0003


In [4]:
import json

# Lendo o CSV e convertendo para uma lista de dicionários.
# Escolhi esse formato porque facilita o acesso aos dados por nome de campo
# nas próximas etapas, especialmente quando for montar o prompt para a IA.
df = pd.read_csv('TOTVS 2026 - SDW2023.csv')
users = df.to_dict(orient='records')

# Adiciono o campo 'news' em cada usuário.
# É aqui que vou guardar as mensagens geradas pela IA na etapa de Transformação.
for user in users:
    user['news'] = []

print(f"{len(users)} clientes carregados.")
print()
print(json.dumps(users, indent=2, ensure_ascii=False))

5 clientes carregados.

[
  {
    "UserID": 1,
    "Nome": "Naruto",
    "Saldo": 1500.0,
    "Limite": 2000.0,
    "Agencia": 1,
    "news": []
  },
  {
    "UserID": 2,
    "Nome": "Hinata",
    "Saldo": 320.5,
    "Limite": 500.0,
    "Agencia": 1,
    "news": []
  },
  {
    "UserID": 3,
    "Nome": "Sasuke",
    "Saldo": 8750.0,
    "Limite": 10000.0,
    "Agencia": 2,
    "news": []
  },
  {
    "UserID": 4,
    "Nome": "Sakura",
    "Saldo": 220.0,
    "Limite": 1000.0,
    "Agencia": 2,
    "news": []
  },
  {
    "UserID": 5,
    "Nome": "Kakashi",
    "Saldo": 15000.0,
    "Limite": 20000.0,
    "Agencia": 3,
    "news": []
  }
]


---

## T — TRANSFORM (Transformação)

Essa é a etapa mais interessante do projeto. Com os dados em mãos, vou gerar uma mensagem de marketing personalizada para cada cliente.

Em vez de chamar uma API externa, criei uma função local que usa o perfil financeiro do cliente para escolher a mensagem mais adequada. O conceito é o mesmo de uma IA generativa: recebo dados de entrada e produzo um texto personalizado como saída.

A lógica de personalização considera o saldo da conta:

- **Saldo abaixo de R$ 500** → cliente que ainda está começando a cuidar das finanças

- **Saldo entre R$ 500 e R$ 5.000** → cliente com perfil intermediário, pronto para diversificar

- **Saldo acima de R$ 5.000** → cliente premium, com interesse em investimentos mais robustos

Cada perfil tem um banco de mensagens diferente, e uma delas é escolhida de forma variada para cada cliente.


### Como seria com uma API de IA (leitura — não executar)

No projeto original do bootcamp, essa etapa usava a API do **GPT-4 (OpenAI)** para gerar as mensagens. O código abaixo mostra exatamente como essa chamada funcionaria — mas **não está sendo executado** porque a conta não tem créditos disponíveis no momento.

Vale entender o que acontece nessa chamada:
- `model`: define qual modelo de linguagem vai processar o prompt
- `messages`: lista com duas entradas — o `system` define o papel da IA (especialista em marketing), e o `user` envia a instrução com os dados do cliente
- `max_tokens`: limita o tamanho da resposta para não gastar mais créditos do que o necessário

A estrutura é a mesma independentemente do modelo usado (GPT, Claude, Gemini etc.) — o que muda é só a biblioteca e o nome do modelo.

```python
# EXEMPLO — não executar (requer créditos na OpenAI)

from openai import OpenAI

client = OpenAI(api_key='SUA_API_KEY_AQUI')

def generate_ai_news_api(user: dict) -> str:
    completion = client.chat.completions.create(
        model='gpt-3.5-turbo',
        messages=[
            {
                'role': 'system',
                'content': 'Você é um especialista em marketing bancário do Santander. '
                           'Crie mensagens curtas, diretas e motivadoras sobre investimentos.'
            },
            {
                'role': 'user',
                'content': f"Crie uma mensagem de marketing para {user['Nome']} "
                           f"sobre a importância dos investimentos. "
                           f"Máximo de 100 caracteres. Responda só com a mensagem, sem aspas."
            }
        ],
        max_tokens=100
    )
    return completion.choices[0].message.content.strip()
```

> **Por que não está rodando?** A conta da OpenAI usada nesse projeto não tem créditos ativos. Para usar, basta adicionar créditos em [platform.openai.com](https://platform.openai.com), substituir a API Key e trocar a chamada `generate_ai_news(user)` por `generate_ai_news_api(user)` no loop abaixo.

---

### Implementação local (em uso)

Como alternativa funcional, criei um banco de mensagens segmentado por perfil financeiro. O resultado final é o mesmo: cada cliente recebe uma mensagem personalizada. A diferença é que aqui o conteúdo é predefinido, não gerado dinamicamente pela IA.


In [5]:
import random

# Banco de mensagens por perfil financeiro.
# Em um projeto real, esse conteúdo seria gerado por uma IA generativa via API.
# Aqui simulo o mesmo comportamento localmente para não depender de créditos externos.
mensagens_por_perfil = {
    "baixo": [
        "{nome}, todo grande investidor começou com pouco. Dê o primeiro passo hoje!",
        "{nome}, investir R$ 50 por mês já faz diferença no longo prazo. Comece agora!",
        "{nome}, seu futuro financeiro começa com uma pequena decisão hoje. Invista!",
    ],
    "moderado": [
        "{nome}, seu saldo está crescendo — hora de diversificar e ir além da poupança!",
        "{nome}, com disciplina e boas escolhas, seu patrimônio pode dobrar em anos.",
        "{nome}, diversificar investimentos é o próximo passo para sua independência!",
    ],
    "premium": [
        "{nome}, seu patrimônio merece uma estratégia avançada. Explore novos ativos!",
        "{nome}, investidores premium fazem o dinheiro trabalhar 24h. Que tal o próximo nível?",
        "{nome}, com seu perfil, fundos exclusivos e renda variável podem acelerar seus ganhos.",
    ]
}


def generate_ai_news(user: dict) -> str:
    """
    Simula a geração de uma mensagem de marketing personalizada.

    Classifica o cliente por perfil financeiro com base no saldo
    e seleciona uma mensagem do banco correspondente.
    A mensagem é limitada a 100 caracteres.
    """
    nome  = user["Nome"]
    saldo = user["Saldo"]

    # Classifica o perfil com base no saldo
    if saldo < 500:
        perfil = "baixo"
    elif saldo < 5000:
        perfil = "moderado"
    else:
        perfil = "premium"

    # Escolhe uma mensagem do banco e substitui o nome do cliente
    mensagem = random.choice(mensagens_por_perfil[perfil])
    mensagem = mensagem.format(nome=nome)

    return mensagem[:100]


print("Gerando mensagens personalizadas...")
print("-" * 60)

for user in users:
    news_text = generate_ai_news(user)

    # Adiciono a mensagem no campo "news" do usuário, com o ícone padrão do projeto
    user["news"].append({
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": news_text
    })

    print(f"{user['Nome']:10s}  ->  {news_text}")

print("-" * 60)
print("Mensagens geradas com sucesso.")


Gerando mensagens personalizadas...
------------------------------------------------------------
Naruto      ->  Naruto, seu saldo está crescendo — hora de diversificar e ir além da poupança!
Hinata      ->  Hinata, investir R$ 50 por mês já faz diferença no longo prazo. Comece agora!
Sasuke      ->  Sasuke, investidores premium fazem o dinheiro trabalhar 24h. Que tal o próximo nível?
Sakura      ->  Sakura, seu futuro financeiro começa com uma pequena decisão hoje. Invista!
Kakashi     ->  Kakashi, investidores premium fazem o dinheiro trabalhar 24h. Que tal o próximo nível?
------------------------------------------------------------
Mensagens geradas com sucesso.


---

## L — LOAD (Carregamento)

Última etapa do pipeline. Agora que os dados foram extraídos e transformados, preciso carregá-los em algum lugar.

No projeto original, essa etapa fazia uma requisição PUT para a API, atualizando o campo `news` de cada usuário no sistema do banco.

Como a API está fora do ar, vou salvar os resultados localmente em dois formatos:

- **CSV**: fácil de abrir no Excel ou Google Sheets para visualizar os resultados
- **JSON**: mantém a estrutura completa dos dados, útil para integrar com outros sistemas

O conceito é o mesmo: estou persistindo os dados transformados em um destino final.

In [6]:
from datetime import datetime

# Monto um novo DataFrame com os dados originais + a mensagem gerada pela IA
resultado = []
for user in users:
    mensagem = user['news'][0]['description'] if user['news'] else ''
    resultado.append({
        'UserID':       user['UserID'],
        'Nome':         user['Nome'],
        'Saldo':        user['Saldo'],
        'Limite':       user['Limite'],
        'Agencia':      user['Agencia'],
        'Mensagem_IA':  mensagem,
        'Data_Geracao': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })

df_resultado = pd.DataFrame(resultado)
df_resultado.to_csv('TOTVS 2026 - SDW2023_resultado.csv', index=False)

print("Arquivo TOTVS 2026 - SDW2023_resultado.csv salvo.")
print()

# Exibo as colunas mais relevantes para conferir o resultado
print(df_resultado[['Nome', 'Saldo', 'Mensagem_IA']].to_string(index=False))

Arquivo TOTVS 2026 - SDW2023_resultado.csv salvo.

   Nome   Saldo                                                                            Mensagem_IA
 Naruto  1500.0         Naruto, seu saldo está crescendo — hora de diversificar e ir além da poupança!
 Hinata   320.5          Hinata, investir R$ 50 por mês já faz diferença no longo prazo. Comece agora!
 Sasuke  8750.0  Sasuke, investidores premium fazem o dinheiro trabalhar 24h. Que tal o próximo nível?
 Sakura   220.0            Sakura, seu futuro financeiro começa com uma pequena decisão hoje. Invista!
Kakashi 15000.0 Kakashi, investidores premium fazem o dinheiro trabalhar 24h. Que tal o próximo nível?


In [7]:
# Salvo também em JSON para preservar a estrutura completa dos dados,
# incluindo o campo 'news' com o ícone e a descrição.
with open('TOTVS 2026 - SDW2023_resultado.json', 'w', encoding='utf-8') as f:
    json.dump(users, f, ensure_ascii=False, indent=2)

print("Arquivo TOTVS 2026 - SDW2023_resultado.json salvo.")
print()
print(json.dumps(users, indent=2, ensure_ascii=False))

Arquivo TOTVS 2026 - SDW2023_resultado.json salvo.

[
  {
    "UserID": 1,
    "Nome": "Naruto",
    "Saldo": 1500.0,
    "Limite": 2000.0,
    "Agencia": 1,
    "news": [
      {
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": "Naruto, seu saldo está crescendo — hora de diversificar e ir além da poupança!"
      }
    ]
  },
  {
    "UserID": 2,
    "Nome": "Hinata",
    "Saldo": 320.5,
    "Limite": 500.0,
    "Agencia": 1,
    "news": [
      {
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": "Hinata, investir R$ 50 por mês já faz diferença no longo prazo. Comece agora!"
      }
    ]
  },
  {
    "UserID": 3,
    "Nome": "Sasuke",
    "Saldo": 8750.0,
    "Limite": 10000.0,
    "Agencia": 2,
    "news": [
      {
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "descri

In [8]:
# Relatório final para confirmar que o pipeline rodou tudo certo
print("=" * 60)
print("         RELATORIO FINAL DO PIPELINE ETL")
print("=" * 60)
print(f"  Clientes processados : {len(users)}")
print(f"  Mensagens geradas    : {sum(len(u['news']) for u in users)}")
print(f"  Arquivos salvos      : SDW2023_resultado.csv e .json")
print(f"  Modelo de IA         : Simulação local (sem API)")
print("=" * 60)
print()
print("Mensagens geradas por cliente:")
print("-" * 60)
for user in users:
    for news in user['news']:
        print(f"  {user['Nome']:10s}  ->  {news['description']}")
print("-" * 60)
print()
print("Pipeline concluido.")

         RELATORIO FINAL DO PIPELINE ETL
  Clientes processados : 5
  Mensagens geradas    : 5
  Arquivos salvos      : SDW2023_resultado.csv e .json
  Modelo de IA         : Simulação local (sem API)

Mensagens geradas por cliente:
------------------------------------------------------------
  Naruto      ->  Naruto, seu saldo está crescendo — hora de diversificar e ir além da poupança!
  Hinata      ->  Hinata, investir R$ 50 por mês já faz diferença no longo prazo. Comece agora!
  Sasuke      ->  Sasuke, investidores premium fazem o dinheiro trabalhar 24h. Que tal o próximo nível?
  Sakura      ->  Sakura, seu futuro financeiro começa com uma pequena decisão hoje. Invista!
  Kakashi     ->  Kakashi, investidores premium fazem o dinheiro trabalhar 24h. Que tal o próximo nível?
------------------------------------------------------------

Pipeline concluido.


---

## Conclusão

O pipeline ETL ficou assim no final:

| Etapa | O que fiz | Ferramenta |
|-------|-----------|------------|
| Extract | Li os dados dos clientes de um arquivo CSV | pandas |
| Transform | Gerei mensagens personalizadas por perfil financeiro usando função local | Simulação local (sem API) |
| Load | Salvei os dados enriquecidos em CSV e JSON | pandas + json |

O principal aprendizado aqui não é sobre uma ferramenta específica — é sobre entender como os dados fluem de uma etapa para a outra. A API, o modelo de IA e o formato de saída podem mudar conforme o projeto. O conceito de ETL permanece o mesmo.

---
*TOTVS - Fundamentos de Engenharia de Dados e Machine Learning — DIO*